# YOLO Dataset Photo Masher
This notebook is the Python equivalent of your C++ Photo Mashing and Label Creation program, optimized for Google Colab.

### Setup Instructions
1. Upload your folders containing the input photos and YOLO `.txt` labels to your Colab environment.
2. Modify the `SETTINGS` block in the first code cell to match your folder paths and desired grid parameters.
3. Run the cells in order.

In [ ]:
import os
import glob
import cv2
import numpy as np
from google.colab import drive

# Optional: Uncomment the line below to mount Google Drive if your datasets are stored there
# drive.mount('/content/drive')

# ==========================
# SETTINGS (Equivalent to Settings.txt and Paths.txt)
# ==========================
IN_PHOTOS_DIR = "./input_photos"
IN_LABELS_DIR = "./input_labels"
OUT_PHOTOS_DIR = "./output_photos"
OUT_LABELS_DIR = "./output_labels"

# Set your desired output resolutions and grid format
RESOLUTION_HORIZONTAL = 1920
RESOLUTION_VERTICAL = 1080
PHOTOS_HORIZONTAL = 2
PHOTOS_VERTICAL = 2
# ==========================

# Create output directories if they don't exist
os.makedirs(OUT_PHOTOS_DIR, exist_ok=True)
os.makedirs(OUT_LABELS_DIR, exist_ok=True)

In [ ]:
def mash_photos_and_create_labels():
    print("Gathering valid images from directory...")
    photo_paths = sorted(glob.glob(os.path.join(IN_PHOTOS_DIR, "*.jpg")) + glob.glob(os.path.join(IN_PHOTOS_DIR, "*.jpeg")))
    
    if not photo_paths:
        print("Found no valid photo files in directory. Please check IN_PHOTOS_DIR.")
        return

    print(f"Found {len(photo_paths)} valid photo files.")
    
    num_per_batch = PHOTOS_HORIZONTAL * PHOTOS_VERTICAL
    res_x_per_photo = RESOLUTION_HORIZONTAL // PHOTOS_HORIZONTAL
    res_y_per_photo = RESOLUTION_VERTICAL // PHOTOS_VERTICAL
    
    current_idx = 0
    batch_count = 0
    valid_photo_paths_local = []
    
    print("\nMashing photos...")
    while current_idx + num_per_batch <= len(photo_paths):
        batch_photos = []
        batch_paths = []
        
        # Read and resize images for the current batch
        for i in range(num_per_batch):
            path = photo_paths[current_idx]
            img = cv2.imread(path)
            if img is not None:
                img = cv2.resize(img, (res_x_per_photo, res_y_per_photo))
                batch_photos.append(img)
                batch_paths.append(path)
            else:
                print(f"Failed to read/resize photo: {path}")
            current_idx += 1
            
        if len(batch_photos) < num_per_batch:
            print("Not enough valid photos for a full batch, skipping remainder.")
            break
            
        valid_photo_paths_local.append(batch_paths)
        
        # Create a blank canvas
        mashed_photo = np.zeros((res_y_per_photo * PHOTOS_VERTICAL, res_x_per_photo * PHOTOS_HORIZONTAL, 3), dtype=np.uint8)
        
        # Paste images into the grid
        for i, img in enumerate(batch_photos):
            x = i % PHOTOS_HORIZONTAL
            y = i // PHOTOS_HORIZONTAL
            
            x_start = x * res_x_per_photo
            y_start = y * res_y_per_photo
            
            mashed_photo[y_start:y_start+res_y_per_photo, x_start:x_start+res_x_per_photo] = img
            
        # Save mashed photo
        batch_name = f"Mashed_{batch_count + 1}"
        out_photo_path = os.path.join(OUT_PHOTOS_DIR, f"{batch_name}.jpg")
        cv2.imwrite(out_photo_path, mashed_photo)
        batch_count += 1
        
    print(f"Successfully mashed {batch_count} image grids. {len(photo_paths) - (batch_count * num_per_batch)} unmashed photos left.")
    
    print("\nCreating scaled YOLO labels...")
    labels_created = 0
    for row_idx, batch_paths in enumerate(valid_photo_paths_local):
        batch_name = f"Mashed_{row_idx + 1}"
        out_label_path = os.path.join(OUT_LABELS_DIR, f"{batch_name}.txt")
        
        with open(out_label_path, 'w') as outfile:
            for col_idx, path in enumerate(batch_paths):
                base = os.path.splitext(os.path.basename(path))[0]
                label_path = os.path.join(IN_LABELS_DIR, f"{base}.txt")
                
                if not os.path.exists(label_path):
                    print(f"No labels found for {path}")
                    continue
                    
                # Determine position in grid
                grid_x = col_idx % PHOTOS_HORIZONTAL
                grid_y = col_idx // PHOTOS_HORIZONTAL
                
                with open(label_path, 'r') as infile:
                    for line in infile:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            try:
                                class_id = int(parts[0])
                                x_center = float(parts[1])
                                y_center = float(parts[2])
                                width = float(parts[3])
                                height = float(parts[4])
                                
                                # Scale coordinates relative to the new mashed image
                                new_x_center = (x_center + grid_x) / PHOTOS_HORIZONTAL
                                new_y_center = (y_center + grid_y) / PHOTOS_VERTICAL
                                new_width = width / PHOTOS_HORIZONTAL
                                new_height = height / PHOTOS_VERTICAL
                                
                                outfile.write(f"{class_id} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}\n")
                            except ValueError:
                                print(f"Skipping malformed line in {label_path}: {line.strip()}")
        labels_created += 1
        
    print(f"Successfully created {labels_created} mashed label files.")

# Execute the workflow
mash_photos_and_create_labels()